# JevBerta Demo

This notebook loads the published Hugging Face model and demonstrates both supported inference APIs:

1. direct `context` + `query` + `choices` prediction
2. JEV-style `state` + `questions` payload prediction

In [1]:
# %pip install -e ..
from pprint import pprint

from jev_berta import JevBerta

MODEL_ID = "leobitz/jev-berta-base-zeroshot-classifier"
model = JevBerta.from_pretrained(MODEL_ID)
MODEL_ID

Fetching 7 files:   0%|          | 0/7 [00:00<?, ?it/s]

The tokenizer you are loading from '/home/leo/.cache/huggingface/hub/models--leobitz--jev-berta-base-zeroshot-classifier/snapshots/d5db5bc920fefdeb9b10d26f947e41eafa604e8f' with an incorrect regex pattern: https://huggingface.co/mistralai/Mistral-Small-3.1-24B-Instruct-2503/discussions/84#69121093e8b480e709447d5e. This will lead to incorrect tokenization. You should set the `fix_mistral_regex=True` flag when loading this tokenizer to fix this issue.


'leobitz/jev-berta-base-zeroshot-classifier'

In [11]:
direct_result = model.predict(
    context="Customer says they were billed twice for the same order and wants a refund.",
    query="What is the best category?",
    choices=["billing", "shipping", "technical", "other"],
)

predicted_label = direct_result['choices'][direct_result['choice_index']]
print(predicted_label, 'with' , direct_result['probabilities'][predicted_label], 'probability')

direct_result = model.predict(
    context="Customer says they were billed twice for the same order and wants a refund.",
    query="What is the best category?",
    choices=["billing", "shipping", "technical", "refund"],
)

predicted_label = direct_result['choices'][direct_result['choice_index']]
print(predicted_label, 'with' , direct_result['probabilities'][predicted_label], 'probability')

other with 0.9995517134666443 probability
refund with 0.8266185522079468 probability


In [15]:
payload = {
    "state": "A customer says they were charged twice for the same order, already emailed support twice without getting a reply, and now wants the duplicate charge refunded immediately.",
    "questions": {
        "refund_requested": {
            "type": "noul",
            "instructions": "Is the customer explicitly asking for a refund?",
            "criteria": {
                "true": "The customer clearly wants money returned or a charge reversed.",
                "false": "The customer is not asking for a refund."
            }
        },
        "owner_team": {
            "type": "choice",
            "instructions": "Which team should take ownership of this case?",
            "criteria": {
                "billing": "Handles duplicate charges, refunds, invoices, and payment disputes.",
                "support": "Handles follow-up communication and general customer assistance.",
                "technical": "Handles bugs, outages, and product malfunctions.",
                "refunding": "Handles cases specifically related to processing refunds."
            }
        },
        "priority": {
            "type": "score",
            "instructions": "How urgent is this case?",
            "criteria": ["Low", "Medium", "High"]
        }
    }
}

jev_result = model.predict_jev(payload)
pprint(jev_result)

{'predictions': {'owner_team': {'choice': 'refunding',
                                'choice_index': 3,
                                'probabilities': {'billing': 0.2700035572052002,
                                                  'refunding': 0.510697066783905,
                                                  'support': 0.2050972878932953,
                                                  'technical': 0.01420213095843792},
                                'query': 'Which team should take ownership of '
                                         'this case?',
                                'rendered_choices': ['billing: Handles '
                                                     'duplicate charges, '
                                                     'refunds, invoices, and '
                                                     'payment disputes.',
                                                     'support: Handles '
                                                     'fo